# CSE 151B Competition Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers â€” for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [31]:
# Install Python dependencies (run once, then skip)
import sys
!{sys.executable} -m pip install -q sympy numpy tqdm requests \
    'antlr4-python3-runtime==4.11.1' ipykernel


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: "'antlr4-python3-runtime==4.11.1'": Expected package name at the start of dependency specifier
    'antlr4-python3-runtime==4.11.1'
    ^


### Run the cell below every time to activate the installed environment. 

In [32]:
# (No venv activation needed â€” packages installed directly into this kernel)
print('Environment ready.')

Environment ready.


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` â€” public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` â€” where per-question results will be written
- `GPU_ID` â€” which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` â€” maximum tokens the model may generate per response

In [33]:
import json
import csv
import re
import sys
from pathlib import Path
from typing import Optional

import requests
from tqdm import tqdm

# Configuration
LM_STUDIO_URL   = "http://localhost:1234"
LM_STUDIO_MODEL = "qwen/qwen3-4b"
DATA_PATH       = "data/private.jsonl"
OUTPUT_PATH     = "results/starter_results.csv"
MAX_TOKENS      = 32768

print(f"LM Studio URL   : {LM_STUDIO_URL}")
print(f"LM Studio model : {LM_STUDIO_MODEL}")
print(f"Max tokens      : {MAX_TOKENS}")

LM Studio URL   : http://localhost:1234
LM Studio model : qwen/qwen3-4b
Max tokens      : 32768


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices â€” present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [34]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n MCQ sample")
print(json.dumps(mcq_sample, indent=2))
print("\nFree-form sample")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

 MCQ sample
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

Free-form sample
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** â€” the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** â€” the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [35]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step.\n\n"

    "═══ FINAL ANSWER FORMAT — THE MOST IMPORTANT RULE ═══\n"
    "At the very LAST LINE of your response, place ALL answers inside exactly ONE \\boxed{}.\n"
    "  DO:     \\boxed{380, 315, 13, 310}  (all parts, comma-separated, one box)\n"
    "  DO:     \\boxed{5/8}  (single answer)\n"
    "  DON'T:  box each sub-answer in a separate \\boxed{} throughout the solution\n"
    "  DON'T:  \\boxed{380}  ...text...  \\boxed{315}  ...text...  \\boxed{13}\n"
    "Even if you use \\boxed{} for intermediate steps during working, you MUST finish with "
    "a single combined \\boxed{a, b, c} on the very last line — all answers, in the order asked.\n"
    "Never leave \\boxed{} empty.\n\n"

    "═══ EXACT FORM RULES ═══\n"
    "1. SYMBOLIC OVER NUMERIC — if the answer is a function applied to given constants, "
    "write the expression, NOT a decimal:\n"
    "  DO:     \\arctan(4.76)          DON'T: 1.3635\n"
    "  DO:     \\ln(0.5)/\\ln(0.96584)  DON'T: 19.94\n"
    "  DO:     (1/2)^{(1999-1963)/31} DON'T: 0.447\n"
    "2. DECIMAL PRECISION — when a decimal is required, give at minimum 6 significant digits:\n"
    "  DO:     7.79744   DON'T: 7.80  |  DO: 442.857   DON'T: 442.86  |  DO: 12.0814  DON'T: 12.08\n"
    "3. PRESERVE STRUCTURE — if the problem writes 2*8*x, write 2*8*x not 16x.\n"
    "4. EXPLICIT MULTIPLICATION — write 3*t*(1-t)^2, not 3t(1-t)^2.\n"
    "5. EXPONENTIALS — write \\exp(0.016*t) or e^{0.016t}, not standalone e^0.016t.\n"
    "6. FRACTIONS — use exact fractions (5/8) for rational results.\n"
    "7. ORDER — answer multi-part questions in the exact order the problem asks.\n"
    "8. NO ANGLE BRACKETS — do not wrap answers in <> brackets.\n\n"

    "Before writing the final \\boxed{}, verify your answer satisfies the original problem. "
    "Commit to your best answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully, then select the single best answer.\n\n"
    "STEP 1 — Solve: Work through the problem step-by-step to derive your answer.\n"
    "STEP 2 — Match: Compare your result against every option:\n"
    "  a) Check algebraic/symbolic equivalence (e.g. pi*sqrt(a) = pi*a^{1/2}, "
    "4/3*ln(3) = 2/3*ln(9), 1-cos^2(x) = sin^2(x)).\n"
    "  b) If options look different, plug in a concrete numeric value for any free variable "
    "and evaluate BOTH your answer and each option — pick the one whose value matches yours.\n"
    "  c) If two options appear numerically equal, prefer the one whose algebraic form "
    "matches your derivation most directly.\n"
    "STEP 3 — Commit: Trust your derivation. Do not abandon a correct answer just because "
    "the option looks different in form.\n\n"
    "You MUST always pick one of the given letters, never say none match. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"{label} user prompt (first 200 chars)")
    print(usr_p[:200], "...\n")

MCQ user prompt (first 200 chars)
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

Free-form user prompt (first 200 chars)
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with Transformers

We load **Qwen3-4B-Thinking-2507** using HuggingFace Transformers with automatic device placement.
- If a GPU is available it will be used automatically.
- On CPU this will be slow (~1-5 min per question) but will work.
- `torch_dtype=torch.float16` halves memory usage vs float32.


In [36]:
# Verify LM Studio is reachable and the model is loaded
try:
    r = requests.get(f"{LM_STUDIO_URL}/v1/models", timeout=5)
    r.raise_for_status()
    models = [m["id"] for m in r.json().get("data", [])]
    if models:
        print(f"LM Studio is running. Loaded models: {models}")
        if LM_STUDIO_MODEL not in models:
            print(f"WARNING: '{LM_STUDIO_MODEL}' not found. Update LM_STUDIO_MODEL to one of: {models}")
    else:
        print("WARNING: LM Studio is running but no model is loaded. Load a model in the Developer tab.")
except requests.exceptions.ConnectionError:
    print(f"ERROR: Cannot reach LM Studio at {LM_STUDIO_URL}")
    print("Make sure LM Studio is open and the server is started (Developer tab, Start Server)")

LM Studio is running. Loaded models: ['qwen/qwen3-4b', 'text-embedding-nomic-embed-text-v1.5']


## 6. Generate Responses

We process each question individually using `model.generate()`.
A progress bar shows estimated time remaining.
Adjust `data[:5]` to run on more questions.


In [37]:
def generate_response(question: str, options=None) -> str:
    system, user = build_prompt(question, options)
    r = requests.post(
        f"{LM_STUDIO_URL}/v1/chat/completions",
        json={
            "model": LM_STUDIO_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            "temperature": 0.2,
            "top_p":       0.85,
            "top_k":       20,
            "max_tokens":  MAX_TOKENS,
        },
        timeout=600,
    )
    if not r.ok:
        raise RuntimeError(f"LM Studio error {r.status_code}: {r.text}")
    return r.json()["choices"][0]["message"]["content"]


#  Resume + Run 
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

done_ids = set()
results  = []
if out_path.exists():
    with open(out_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            done_ids.add(int(row['id']))
            results.append({'id': int(row['id']), 'response': row['response']})
    print(f"Resuming: {len(done_ids)} done, {len(data) - len(done_ids)} remaining")
else:
    print("Starting fresh run")

write_header = not out_path.exists()
with open(out_path, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'response'])
    if write_header:
        writer.writeheader()
    for item in tqdm(data, desc="Generating"):
        if item['id'] in done_ids:
            continue
        try:
            response = generate_response(item["question"], item.get("options"))
        except Exception as e:
            print(f"\nERROR on id={item['id']}: {e}")
            record = {'id': item['id'], 'response': f"ERROR: {e}"}
            results.append(record)
            writer.writerow(record)
            f.flush()
            continue
        record = {'id': item['id'], 'response': response}
        results.append(record)
        writer.writerow(record)
        f.flush()

Resuming: 943 done, 0 remaining


Generating: 100%|██████████| 943/943 [00:00<00:00, 3279625.76it/s]


In [41]:
# ── Rerun failed submissions with increased MAX_TOKENS ───────────────────────
RETRY_MAX_TOKENS = 32768

out_path = Path(OUTPUT_PATH)

# Load current results and identify failed rows
current_rows = []
with open(out_path, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        current_rows.append({'id': int(row['id']), 'response': row['response']})

failed_ids = {r['id'] for r in current_rows if r['response'].startswith('ERROR:') or not r['response'].strip()}
print(f'Found {len(failed_ids)} failed submissions to retry: {sorted(failed_ids)}')

# Build lookups
data_by_id = {item['id']: item for item in data}
row_index   = {r['id']: i for i, r in enumerate(current_rows)}

def flush_csv():
    with open(out_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['id', 'response'])
        writer.writeheader()
        writer.writerows(current_rows)

# Retry each failed item, writing to CSV after each one
n_retried = 0
for fid in tqdm(sorted(failed_ids), desc='Retrying failed'):
    item = data_by_id[fid]
    try:
        system, user = build_prompt(item['question'], item.get('options'))
        r = requests.post(
            f'{LM_STUDIO_URL}/v1/chat/completions',
            json={
                'model': LM_STUDIO_MODEL,
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user',   'content': user},
                ],
                'temperature': 0.2,
                'top_p':       0.85,
                'top_k':       20,
                'max_tokens':  RETRY_MAX_TOKENS,
            },
            timeout=600,
        )
        if not r.ok:
            raise RuntimeError(f'LM Studio error {r.status_code}: {r.text}')
        msg      = r.json()['choices'][0]['message']
        response = msg.get('content', '')
        # Fallback: if the model used all tokens thinking and content is empty,
        # extract a \boxed{} answer from reasoning_content if one exists there
        if not response.strip():
            reasoning = msg.get('reasoning_content', '')
            boxed = re.findall(r'\boxed\{[^}]*\}', reasoning)
            response = boxed[-1] if boxed else ''
    except Exception as e:
        print(f'\nERROR on retry id={fid}: {e}')
        response = f'ERROR: {e}'

    current_rows[row_index[fid]]['response'] = response
    n_retried += 1
    print(f'  id={fid}: {"empty" if not response.strip() else "OK"} ({len(response)} chars)')
    flush_csv()

still_failed = sum(1 for r in current_rows if r['response'].startswith('ERROR:') or not r['response'].strip())
print(f'\nDone. {n_retried} retried, {still_failed} still failed. Results saved to {out_path}')


Found 1 failed submissions to retry: [412]


Retrying failed: 100%|██████████| 1/1 [00:17<00:00, 17.20s/it]

  id=412: OK (1582 chars)

Done. 1 retried, 0 still failed. Results saved to results\starter_results.csv


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [39]:
'''
def extract_letter(text: str) -> str:
    m = re.search(r'\\boxed\{([A-Za-z])\}', text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r'\b([A-Z])\b', text.upper())
    return matches[-1] if matches else ''


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, '.')
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in zip(subset, responses):
    is_mcq = bool(item.get('options'))
    gold   = item['answer']

    if is_mcq:
        correct = score_mcq(response, gold)
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        correct = judger.auto_judge(
            response,
            gold_list,
            options=[[]]*len(gold_list),
        )

    results.append({
        'id':       item['id'],
        'is_mcq':   is_mcq,
        'gold':     gold,
        'response': response,
        'correct':  correct,
    })
    print(f"id={item['id']} correct={correct}")
    '''


<>:3: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:3: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
C:\Users\Adam\AppData\Local\Temp\ipykernel_16944\4093079485.py:3: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  m = re.search(r'\\boxed\{([A-Za-z])\}', text)


'\ndef extract_letter(text: str) -> str:\n    m = re.search(r\'\\boxed\\{([A-Za-z])\\}\', text)\n    if m:\n        return m.group(1).upper()\n    matches = re.findall(r\'\x08([A-Z])\x08\', text.upper())\n    return matches[-1] if matches else \'\'\n\n\ndef score_mcq(response: str, gold_letter: str) -> bool:\n    return extract_letter(response) == gold_letter.strip().upper()\n\n\n# Load Judger for free-form scoring\nsys.path.insert(0, \'.\')\nfrom judger import Judger\njudger = Judger(strict_extract=False)\n\nresults = []\nfor item, response in zip(subset, responses):\n    is_mcq = bool(item.get(\'options\'))\n    gold   = item[\'answer\']\n\n    if is_mcq:\n        correct = score_mcq(response, gold)\n    else:\n        gold_list = gold if isinstance(gold, list) else [gold]\n        correct = judger.auto_judge(\n            response,\n            gold_list,\n            options=[[]]*len(gold_list),\n        )\n\n    results.append({\n        \'id\':       item[\'id\'],\n        \'is_m

## 8. Summary

Print accuracy broken down by question type.

In [40]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

KeyError: 'is_mcq'

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set â€” you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set â€” no ground-truth available):  
Each line: `{id, is_mcq, response}` â€” omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'response'])
    writer.writeheader()
    for r in results:
        writer.writerow({'id': r['id'], 'response': r['response']})

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** â€” try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** â€” adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** â€” the competition allows model fine-tuning; see the course resources for guidance

Good luck!